## Extractor Configuration and Testing ##

This is the notebook for configuring and testing the Extractor. 

Please first consider (together with Gemini) which categories need to be captured so that your test orders are recorded accurately. 

Minimum requirements: It should be possible to process orders containing multiple pizzas. 

Please start by using Anthropic Haiku via the API; later, you can try to see if local models also produce good results. 


In [1]:
from dotenv import load_dotenv
import os
import getpass

load_dotenv()


if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Introduce tu  AWS Key: ")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("laude-haiku-4-5-20251001", model_provider="anthropic", temperature=0.0)

### Here is the basic structure for the Extractor: ###

We need a **system prompt** that explains to the model what it needs to do.

The respective order is provided as **Human Massage**. 

In [3]:
#Example for a basic system prompt
from langchain_core.prompts import ChatPromptTemplate

model = init_chat_model("claude-haiku-4-5-20251001", model_provider="", temperature=0.0)

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "Parse a customer's pizza order into valid JSON with the keys number_pizzas, pizza_name, size, crust, ingredients, added_ingredients, removed_ingredients."
            "- If a customer orders a pizza by its standard name, infer and populate its default classic ingredients even if the customer does not explicitly list them:\n"
            "  * Hawaiian: pineapple, ham, tomato sauce, mozzarella cheese.\n"
            "  * Margherita: tomato sauce, mozzarella cheese, fresh basil.\n"
            "  * Pepperoni: tomato sauce, mozzarella cheese, pepperoni.\n"
            "  * Four Cheese (Quattro Formaggi): mozzarella, gorgonzola, parmesan, provolone.\n"
            "  * BBQ / Barbecue: BBQ sauce, mozzarella, chicken or beef, onions.\n"
            "- If the customer explicitly adds, replaces, or removes ingredients (e.g., 'no onions', 'add mushrooms'), adjust the ingredient list accordingly."
        ),
        #MessagesPlaceholder("examples"),
        
        ("human", "{order_text}"),
    ]
)

In [4]:
order = "I would like a large Hawaian pizza with cheese crust, with extra cheese and pineapple, without tomato sauce."
prompt = prompt_template.invoke({"order_text": order})
res = model.invoke(prompt)
res.content


'```json\n{\n  "number_pizzas": 1,\n  "pizza_name": "Hawaiian",\n  "size": "large",\n  "crust": "cheese crust",\n  "ingredients": [\n    "pineapple",\n    "ham",\n    "mozzarella cheese"\n  ],\n  "added_ingredients": [\n    "extra cheese",\n    "extra pineapple"\n  ],\n  "removed_ingredients": [\n    "tomato sauce"\n  ]\n}\n```'

Then we need to define the **extraction schema**. Try using **Pydantic** first, then try to use a **JSON Schema** with mode=json (preferably converted by Gemini).

Here is a simple schema to get started:

In [11]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Information about a the ordered pizza."""
    
    number_pizzas: int = Field(..., description="The number of pizzas")
    pizza_name: str = Field(
        ..., description="The name of the pizza if provided, else None."
    )
    size: Literal["small", "normal", "large"] = Field(
        ..., description="The size of the pizza"
    )
    crust: Literal["normal", "with cheese", "big"] = Field(..., description="The crust of the pizza")    
    added_ingredients: List[str] = Field(
        default=None, description="The extra ingredients for the pizza."
    )
    removed_ingredients: List[str] = Field(
            default=None, description="The removed ingredients for the pizza."
    )
    ingredients: List[str] = Field(..., description="The ingredients provided by the pizza's name.")
    


class Pizza_order(BaseModel):
    """Extracted data about pizzas."""
    # Creates a model so that we can extract multiple entities.
    order: List[Pizza]


structured_llm = model.with_structured_output(schema=Pizza_order)



In [13]:
order = "I would like a large Hawaian pizza with cheese crust, with extra cheese and pineapple, without tomato sauce."
prompt = prompt_template.invoke({"order_text": order})
res = structured_llm.invoke(prompt)
res.model_dump()

{'order': [{'number_pizzas': 1,
   'pizza_name': 'Hawaiian',
   'size': 'large',
   'crust': 'with cheese',
   'added_ingredients': ['extra cheese', 'pineapple'],
   'removed_ingredients': ['tomato sauce'],
   'ingredients': ['pineapple', 'ham', 'mozzarella cheese']}]}

If necesary, you can add more examples to the prompt template to improve the extraction accuracy.

In [17]:
#Creat a list of few-shot massege with the original examples form the Google Whitepaper

examples = [
    {"role": "user", "content": "I want a small margherita"},
    {"role": "assistant", "content": '{"size": "small","type": "normal", "ingredients": [], "added_ingredients": [], "removed_ingredients": [], "number_pizzas": 1, "crust": "normal" }'},
    {"role": "user", "content": "I would like a large Hawaian pizza with cheese crust, with extra cheese and pineapple, without tomato sauce"},
    {"role": "assistant", "content": '{"number_pizzas": 1,"pizza_name": "Hawaiian","size": "large","crust": "with cheese","added_ingredients": ["extra cheese", "pineapple"],"removed_ingredients": ["tomato sauce"],"ingredients": ["pineapple", "ham", "mozzarella cheese"]}'},
]    

In [18]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template_with_examples = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "Parse a customer's pizza order into valid JSON with the keys number_pizzas, pizza_name, size, crust, ingredients, added_ingredients, removed_ingredients."
            "- If a customer orders a pizza by its standard name, infer and populate its default classic ingredients even if the customer does not explicitly list them:\n"
            "  * Hawaiian: pineapple, ham, tomato sauce, mozzarella cheese.\n"
            "  * Margherita: tomato sauce, mozzarella cheese, fresh basil.\n"
            "  * Pepperoni: tomato sauce, mozzarella cheese, pepperoni.\n"
            "  * Four Cheese (Quattro Formaggi): mozzarella, gorgonzola, parmesan, provolone.\n"
            "  * BBQ / Barbecue: BBQ sauce, mozzarella, chicken or beef, onions.\n"
            "- If the customer explicitly adds, replaces, or removes ingredients (e.g., 'no onions', 'add mushrooms'), adjust the ingredient list accordingly."
        ),
        MessagesPlaceholder("examples"),
        
        ("human", "{order_text}"),
    ]
)

In [21]:
text = "I would like a small BBQ without mozzarella"
prompt = prompt_template_with_examples.invoke({"examples": examples, "order_text": text})

for message in prompt.messages:
    message.pretty_print()

================================ System Message ================================

You are an expert extraction algorithm. Only extract relevant information from the text. Parse a customer's pizza order into valid JSON with the keys number_pizzas, pizza_name, size, crust, ingredients, added_ingredients, removed_ingredients.- If a customer orders a pizza by its standard name, infer and populate its default classic ingredients even if the customer does not explicitly list them:
  * Hawaiian: pineapple, ham, tomato sauce, mozzarella cheese.
  * Margherita: tomato sauce, mozzarella cheese, fresh basil.
  * Pepperoni: tomato sauce, mozzarella cheese, pepperoni.
  * Four Cheese (Quattro Formaggi): mozzarella, gorgonzola, parmesan, provolone.
  * BBQ / Barbecue: BBQ sauce, mozzarella, chicken or beef, onions.
- If the customer explicitly adds, replaces, or removes ingredients (e.g., 'no onions', 'add mushrooms'), adjust the ingredient list accordingly.
================================ Human Me

In [22]:
res = model.invoke(prompt)
res.pretty_print()

================================== Ai Message ==================================

```json
{
  "number_pizzas": 1,
  "pizza_name": "BBQ",
  "size": "small",
  "crust": "normal",
  "ingredients": ["BBQ sauce", "chicken", "onions"],
  "added_ingredients": [],
  "removed_ingredients": ["mozzarella"]
}
```
